In [12]:
import os
import requests
import numpy as np
import soundfile as sf
import gradio as gr
from IPython.display import Audio, display
from kokoro import KPipeline
from faster_whisper import WhisperModel

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5:1.5b"
KOKORO_LANG_CODE = "a"
KOKORO_VOICE = "af_heart"
SAMPLE_RATE = 24000

print("Imports completed.")

Imports completed.


In [13]:
def ask_ollama(question: str) -> str:
    question = (question or "").strip()
    if not question:
        return "Please enter a question."

    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": question}],
            "stream": False,
        },
        timeout=180,
    )
    response.raise_for_status()
    return response.json()["message"]["content"].strip()

print(ask_ollama("Introduce yourself in one short sentence."))

I am an AI designed to assist and provide information on various topics efficiently and accurately.


In [14]:

def ask_ollama(question: str) -> str:
    question = (question or "").strip()

    if not question:
        return "Please enter a question."

    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are EchoAI, a helpful voice assistant. "
                        "Answer clearly and concisely in 2-3 sentences. "
                        "Avoid unnecessary details."
                    )
                },
                {
                    "role": "user",
                    "content": question
                }
            ],
            "stream": False,
            "options": {
                "num_predict": 100
            }
        },
        timeout=180,
    )

    response.raise_for_status()
    return response.json()["message"]["content"].strip()


print(ask_ollama("Introduce yourself in one short sentence."))


I am EchoAI - your personalized virtual assistant ready to help with tasks, answer questions, and more.


In [15]:
kokoro_pipeline = KPipeline(lang_code=KOKORO_LANG_CODE)

def text_to_speech(text: str) -> str:
    text = (text or "").strip()
    if not text:
        raise ValueError("There is no text to speak.")

    chunks = []
    for _graphemes, _phonemes, audio_chunk in kokoro_pipeline(
        text, voice=KOKORO_VOICE
    ):
        chunks.append(np.asarray(audio_chunk, dtype=np.float32))

    if not chunks:
        raise RuntimeError("Kokoro did not produce any audio.")

    audio = np.concatenate(chunks)
    output_path = os.path.abspath("echoai_response.wav")
    sf.write(output_path, audio, SAMPLE_RATE)
    return output_path

test_audio_path = text_to_speech("Hello! This is EchoAI speaking.")
display(Audio(filename=test_audio_path))

c:\Users\anuna\OneDrive\Desktop\PYSPIDERS\ECHO_AI\.venv311\Lib\site-packages\torch\nn\modules\rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
c:\Users\anuna\OneDrive\Desktop\PYSPIDERS\ECHO_AI\.venv311\Lib\site-packages\torch\nn\utils\weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [16]:
WHISPER_MODEL_SIZE = "small"
whisper_model = WhisperModel(
    WHISPER_MODEL_SIZE,
    device="cpu",
    compute_type="int8",
)

def transcribe_audio(audio_path: str) -> str:
    if not audio_path:
        return ""
    segments, _info = whisper_model.transcribe(audio_path, vad_filter=True)
    return " ".join(segment.text.strip() for segment in segments).strip()

print("Faster-Whisper is ready.")

Faster-Whisper is ready.


In [17]:
def run_assistant(question: str, audio_path: str):
    typed_question = (question or "").strip()
    recognized_question = transcribe_audio(audio_path) if audio_path else ""
    final_question = typed_question or recognized_question

    if not final_question:
        return "", "Please type a question or record/upload audio.", None

    try:
        answer = ask_ollama(final_question)
        answer_audio_path = text_to_speech(answer)
        return final_question, answer, answer_audio_path
    except requests.RequestException as exc:
        return final_question, f"Could not connect to Ollama: {exc}", None
    except Exception as exc:
        return final_question, f"Something went wrong: {exc}", None

In [18]:
'''import time

test_question = "What is phishing? Explain in two short sentences."

# Test 1: Ollama / Qwen
start = time.perf_counter()
test_answer = ask_ollama(test_question)
ollama_time = time.perf_counter() - start

print(f"Qwen time: {ollama_time:.2f} seconds")
print("Answer:", test_answer)

# Test 2: Kokoro TTS
start = time.perf_counter()
test_audio = text_to_speech(test_answer)
kokoro_time = time.perf_counter() - start

print(f"\nKokoro time: {kokoro_time:.2f} seconds")
print(f"Total time: {ollama_time + kokoro_time:.2f} seconds")'''

'import time\n\ntest_question = "What is phishing? Explain in two short sentences."\n\n# Test 1: Ollama / Qwen\nstart = time.perf_counter()\ntest_answer = ask_ollama(test_question)\nollama_time = time.perf_counter() - start\n\nprint(f"Qwen time: {ollama_time:.2f} seconds")\nprint("Answer:", test_answer)\n\n# Test 2: Kokoro TTS\nstart = time.perf_counter()\ntest_audio = text_to_speech(test_answer)\nkokoro_time = time.perf_counter() - start\n\nprint(f"\nKokoro time: {kokoro_time:.2f} seconds")\nprint(f"Total time: {ollama_time + kokoro_time:.2f} seconds")'

In [19]:
import time
test_question = "What is phishing? Explain in two short sentences."

start = time.perf_counter()
test_answer = ask_ollama(test_question)
ollama_time = time.perf_counter() - start

print(f"Qwen time: {ollama_time:.2f} seconds")
print("Answer:", test_answer)

start = time.perf_counter()
test_audio = text_to_speech(test_answer)
kokoro_time = time.perf_counter() - start

print(f"Kokoro time: {kokoro_time:.2f} seconds")
print(f"Total time: {ollama_time + kokoro_time:.2f} seconds")

Qwen time: 8.67 seconds
Answer: Phishing is a cybercrime where attackers send deceptive emails or instant messages to trick individuals into providing personal information such as passwords, credit card numbers, or other sensitive data.
Kokoro time: 14.75 seconds
Total time: 23.42 seconds


In [20]:
import time

short_text = "Hello! How can I help you?"

start = time.perf_counter()
short_audio = text_to_speech(short_text)
short_time = time.perf_counter() - start

print(f"Short speech generation: {short_time:.2f} seconds")
display(Audio(filename=short_audio))

Short speech generation: 2.90 seconds


In [23]:
import gradio as gr
print(gr.__version__)

6.30.0


In [ ]:
with gr.Blocks(title="EchoAI Voice Assistant") as demo:
    gr.Markdown("# EchoAI")
    gr.Markdown("Type or speak a question. Qwen answers, and Kokoro reads the answer aloud.")

    with gr.Row():
        question_box = gr.Textbox(
            label="Type your question (optional if audio is provided)",
            placeholder="Ask EchoAI something...",
            lines=3,
        )
        
        audio_input = gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label="Record or upload audio",
        )

    run_button = gr.Button("Ask EchoAI", variant="primary")
    understood_output = gr.Textbox(label="Question understood", lines=2)
    answer_output = gr.Textbox(label="AI answer", lines=5)
    speech_output = gr.Audio(label="Spoken answer", type="filepath")

    run_button.click(
        fn=run_assistant,
        inputs=[question_box, audio_input],
        outputs=[understood_output, answer_output, speech_output],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
